登录

In [2]:
import requests
import json
from os.path import expanduser
from requests.auth import HTTPBasicAuth

def sign_in():
    # load credentials
    with open(expanduser('brain_credentials.txt')) as f:
        credentials = json.load(f)

    # Extract username and password from the list
    username, password = credentials

    # Create a session object
    sess = requests.Session()

    # Set up basic authentication
    sess.auth = HTTPBasicAuth(username, password)

    # Send a POST request to the API for authentication
    response = sess.post('https://api.worldquantbrain.com/authentication')

    # Print response status and content for debugging
    print(response.status_code)
    print(response.json())

    return sess

sess = sign_in()

201
{'user': {'id': 'KY55225'}, 'token': {'expiry': 14400.0}, 'permissions': ['TUTORIAL', 'WORKDAY']}


获取数据集ID为fundamental6 (Company Fundamental Data for Equity)下的所有数据字段

In [3]:
### Get Datafield like Data Explorer 获取所有满足条件的数据字段及其ID
def get_datafields(
        s,
        searchScope,
        dataset_id: str = '',
        search: str = ''
):
        import pandas as pd
        instrument_type = searchScope['instrumentType']
        region = searchScope['region']
        delay = searchScope['delay']
        universe = searchScope['universe']
        if len(search) == 0:
            url_template = "https://api.worldquantbrain.com/data-fields?" +\
                f"&instrumentType={instrument_type}" +\
                f"&region={region}&delay={str(delay)}&universe={universe}&dataset.id={dataset_id}&limit=50" +\
                "&offset={x}"
            count = s.get(url_template.format(x=0)).json()['count']
        else:
            url_template = "https://api.worldquantbrain.com/data-fields?" +\
                f"&instrumentType={instrument_type}" +\
                f"&region={region}&delay={str(delay)}&universe={universe}&limit=50" +\
                f"search={search}" +\
                "&offset={x}"
            count = 100

        datafields_list =[]
        for x in range(0, count, 50):
            datafields = s.get(url_template.format(x=x))
            datafields_list.append(datafields.json()['results'])
        datafields_list_flat = [item for sublist in datafields_list for item in sublist]

        datafields_df = pd.DataFrame(datafields_list_flat)
        return datafields_df

In [4]:
searchScope = {'region': 'USA', 'delay': '1', 'universe': 'TOP3000', 'instrumentType': 'EQUITY'}
fnd6 = get_datafields(s=sess, searchScope=searchScope, dataset_id='fundamental6')
fnd6 = fnd6[fnd6['type']=="MATRIX"]

In [5]:
datafields_list_fnd6 = fnd6['id'].values
datafields_list_fnd6

array(['assets', 'assets_curr', 'bookvalue_ps', 'capex', 'cash',
       'cash_st', 'cashflow', 'cashflow_dividends', 'cashflow_fin',
       'cashflow_invst', 'cashflow_op', 'cogs', 'current_ratio', 'debt',
       'debt_lt', 'debt_st', 'depre_amort', 'ebit', 'ebitda', 'employee',
       'enterprise_value', 'eps', 'equity', 'fnd6_acdo', 'fnd6_acodo',
       'fnd6_acox', 'fnd6_acqgdwl', 'fnd6_acqintan',
       'fnd6_adesinda_curcd', 'fnd6_aldo', 'fnd6_am', 'fnd6_aodo',
       'fnd6_aox', 'fnd6_aqc', 'fnd6_aqi', 'fnd6_aqs', 'fnd6_beta',
       'fnd6_capxv', 'fnd6_ceql', 'fnd6_ch', 'fnd6_ci', 'fnd6_cibegni',
       'fnd6_cicurr', 'fnd6_cidergl', 'fnd6_cik', 'fnd6_cimii',
       'fnd6_ciother', 'fnd6_cipen', 'fnd6_cisecgl', 'fnd6_citotal',
       'fnd6_city', 'fnd6_cld2', 'fnd6_cld3', 'fnd6_cld4', 'fnd6_cld5',
       'fnd6_cptmfmq_actq', 'fnd6_cptmfmq_atq', 'fnd6_cptmfmq_ceqq',
       'fnd6_cptmfmq_dlttq', 'fnd6_cptmfmq_dpq', 'fnd6_cptmfmq_lctq',
       'fnd6_cptmfmq_oibdpq', 'fnd6_cptmfmq_o

将datafield和operator替换到Alpha模板 (框架) 中 ☞ group_rank(ts_rank({fundamental model data}, 252), industry), 批量生成Alpha

In [6]:
# 模板☞<group_compare_ops>(<ts_compare_ops>(<company_fundamentals>, <days>), <group>)

In [7]:
group_compare_op = ['group_rank', 'group_zscore', 'group_neutralize']
ts_compare_op = ['ts_rank', 'ts_zscore', 'ts_quantile']
company_fundamentals = datafields_list_fnd6
days = [600, 200]
group = ['market', 'industry', 'subindustry', 'sector']

alpha_expressions = []

for gco in group_compare_op:
    for tco in ts_compare_op:
        for cf in company_fundamentals:
            for d in days:
                for grp in group:
                    alpha_expressions.append(f'{gco} ( {tco}({cf}, {d}) , {grp})')

# Print or return the result strings list
print(f'there are total {len(alpha_expressions)} alpha expressions')

there are total 41328 alpha expressions


In [8]:
alpha_expressions[:20]

['group_rank ( ts_rank(assets, 600) , market)',
 'group_rank ( ts_rank(assets, 600) , industry)',
 'group_rank ( ts_rank(assets, 600) , subindustry)',
 'group_rank ( ts_rank(assets, 600) , sector)',
 'group_rank ( ts_rank(assets, 200) , market)',
 'group_rank ( ts_rank(assets, 200) , industry)',
 'group_rank ( ts_rank(assets, 200) , subindustry)',
 'group_rank ( ts_rank(assets, 200) , sector)',
 'group_rank ( ts_rank(assets_curr, 600) , market)',
 'group_rank ( ts_rank(assets_curr, 600) , industry)',
 'group_rank ( ts_rank(assets_curr, 600) , subindustry)',
 'group_rank ( ts_rank(assets_curr, 600) , sector)',
 'group_rank ( ts_rank(assets_curr, 200) , market)',
 'group_rank ( ts_rank(assets_curr, 200) , industry)',
 'group_rank ( ts_rank(assets_curr, 200) , subindustry)',
 'group_rank ( ts_rank(assets_curr, 200) , sector)',
 'group_rank ( ts_rank(bookvalue_ps, 600) , market)',
 'group_rank ( ts_rank(bookvalue_ps, 600) , industry)',
 'group_rank ( ts_rank(bookvalue_ps, 600) , subindustr

In [9]:
alpha_list = []

for alpha_expression in alpha_expressions:
    # print('正在将如下Alpha表达式与setting封装')
    simulation_data = {
    'type': 'REGULAR',
    'settings': {
        'instrumentType': 'EQUITY',
        'region': 'USA',
        'universe': 'TOP3000',
        'delay': 1,
        'decay': 6,
        'neutralization': 'SUBINDUSTRY',
        'truncation': 0.08,
        'pasteurization': 'ON',
        'unitHandling': 'VERIFY',
        'nanHandling': 'ON',
        'language': 'FASTEXPR',
        'visualization': False,
        },
    'regular': alpha_expression ##表达式
    }

    alpha_list.append(simulation_data)
print(f'there are {len(alpha_list)}  Alphas to simulate')

there are 41328  Alphas to simulate


In [12]:
alpha_list[2872]

{'type': 'REGULAR',
 'settings': {'instrumentType': 'EQUITY',
  'region': 'USA',
  'universe': 'TOP3000',
  'delay': 1,
  'decay': 6,
  'neutralization': 'SUBINDUSTRY',
  'truncation': 0.08,
  'pasteurization': 'ON',
  'unitHandling': 'VERIFY',
  'nanHandling': 'ON',
  'language': 'FASTEXPR',
  'visualization': False},
 'regular': 'group_rank ( ts_rank(fnd6_newqv1300_ibcomq, 600) , market)'}

将Alpha一个一个发送至服务器进行回测，并检查是否断线，如断线则重连

设置log

In [13]:
import logging
# Configure the logging settings
logging.basicConfig(filename='simulation_day3.log', level=logging.INFO,
                    format='%(asctime)s - %(levelname)s - %(message)s')

In [ ]:
from time import sleep
import logging

alpha_fail_attempt_tolerance = 15  # 每个 alpha 允许的最大失败尝试次数

for alpha in alpha_list[2872:]:
    keep_trying = True      # 控制 while 循环重试的标志
    failure_count = 0       # 记录失败尝试次数的计数器

    while keep_trying:
        try:
            # 尝试发送模拟请求
            sim_resp = sess.post(
                'https://api.worldquantbrain.com/simulations',
                json=alpha,     # 将当前 alpha (一个JSON) 发送到服务器 
            )
            sim_progress_url = sim_resp.headers['Location']
            logging.info(f'Alpha location is: {sim_progress_url}')
            print(f'Alpha location is: {sim_progress_url}')
            keep_trying = False

        except Exception as e:
            # 处理异常: 记录错误, 休眠并增加失败次数计数
            logging.error(f'No location, sleep 15 and retry, error message: {str(e)}')
            print('No location, sleep 15 and retry')
            sleep(15)
            failure_count += 1

            if failure_count >= alpha_fail_attempt_tolerance:
                sess = sign_in()
                failure_count = 0
                # 记录并打印消息, 指示将移动到下一个 alpha
                logging.error(f'No location for too many times, move to next alpha {alpha['regular']}')
                print(f'No location for too many times, move to next alpha {alpha['regular']}')
                break


Alpha location is: https://api.worldquantbrain.com/simulations/RXkA36MP4meaG1i0461v0E
Alpha location is: https://api.worldquantbrain.com/simulations/4BK9YObQn4l0cfy1fazkcsD9
Alpha location is: https://api.worldquantbrain.com/simulations/2Nq7Pgwn4Jz8BB1h2VHwnvL
No location, sleep 15 and retry
No location, sleep 15 and retry
No location, sleep 15 and retry
No location, sleep 15 and retry
No location, sleep 15 and retry
Alpha location is: https://api.worldquantbrain.com/simulations/3ktpkda9J4RQbe3KFcIMVUy
No location, sleep 15 and retry
No location, sleep 15 and retry
No location, sleep 15 and retry
No location, sleep 15 and retry
No location, sleep 15 and retry
Alpha location is: https://api.worldquantbrain.com/simulations/7i50fcwN4wF8Lih3S0RMya
No location, sleep 15 and retry
No location, sleep 15 and retry
No location, sleep 15 and retry
Alpha location is: https://api.worldquantbrain.com/simulations/17SXgLaif4tya19T9yzgnTl
No location, sleep 15 and retry
No location, sleep 15 and retry